# Chat Mode — Integration Test

Tests the full chat pipeline across multiple questions:
- **Q1:** "How frequent were positive events (e.g. limit increases) in the last 18 months?"
- **Follow-up:** "Are the positive events consistent with the bureau scores?" (brings in bureau)
- **Q2 (independent):** "What does the recent 3 months payment pattern look like?"

In [ ]:
%load_ext dotenv
%dotenv

%load_ext autoreload
%autoreload 2

import os, sys, json
import nest_asyncio
nest_asyncio.apply()

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) != 'notebooks':
    PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'simulated')
PROFILE_DIR = os.path.join(PROJECT_ROOT, 'config', 'data_profiles')
PILLAR_DIR = os.path.join(PROJECT_ROOT, 'config', 'pillars')

# ═══════════════ CONFIG ═══════════════
SAFECHAIN_MODEL = "gpt-4.1"
CASE_ID = "CASE-00001"
PILLAR = "credit_risk"
MODE = "chat"
QUESTION = "How frequent were positive events (e.g. limit increases) in the last 18 months?"
# ═════════════════════════════════════

print(f'Case: {CASE_ID} | Pillar: {PILLAR} | Mode: {MODE}')
print(f'Question: {QUESTION}')

## 1. Load Data

In [ ]:
from data.gateway import SimulatedDataGateway
from data.catalog import DataCatalog
from tools.data_tools import init_tools

gw = SimulatedDataGateway.from_case_folders(DATA_DIR)
catalog = DataCatalog(profile_dir=PROFILE_DIR)
init_tools(gw, catalog)
gw.set_case(CASE_ID)

print(f'Tables: {gw.list_tables()}')

## 2. Preview Relevant Data

In [ ]:
# positive_events from model scores
rows = gw.query('model_scores')
if rows:
    for i, row in enumerate(rows[:3]):
        print(f'Row {i}: trans_month={row.get("trans_month", "N/A")}, positive_events={row.get("positive_events", "N/A")}')
print()

tenure = gw.query('cust_tenure')
if tenure:
    print(f'Tenure: {tenure[0]}')

## 3. Connect LLM

In [ ]:
from gateway.safechain_adapter import SafeChainAdapter

try:
    from safechain.lcel import model as safechain_model
    llm = safechain_model(SAFECHAIN_MODEL)
    adapter = SafeChainAdapter(llm=llm, model_name=SAFECHAIN_MODEL)
    print(f'SafeChain adapter: model={SAFECHAIN_MODEL}')
except ImportError:
    from gateway.openai_adapter import OpenAIAdapter
    adapter = OpenAIAdapter(model='gpt-4.1')
    print(f'OpenAI adapter (SafeChain not available)')

## 4. Pipeline Setup

In [ ]:
from gateway.firewall_stack import FirewallStack
from log.event_logger import EventLogger
from agents.session_registry import SessionRegistry
from agents.general_specialist import GeneralSpecialist
from orchestrator.orchestrator import Orchestrator
from orchestrator.team import TeamConstructor
from orchestrator.chat_agent import ChatAgent
from config.pillar_loader import PillarLoader
from skills.domain.loader import load_domain_skill, list_domain_skills

logger = EventLogger(session_id=f'chat-{CASE_ID}', log_dir=os.path.join(PROJECT_ROOT, 'logs'))
pillar_loader = PillarLoader(pillar_dir=PILLAR_DIR)
pillar_config = pillar_loader.load(PILLAR) or {}
registry = SessionRegistry()

# Shared across all questions — these can be reused
general = GeneralSpecialist(firewall=FirewallStack(adapter=adapter, logger=logger), logger=logger)
orchestrator = Orchestrator(FirewallStack(adapter=adapter, logger=logger), logger, registry, PILLAR, pillar_config=pillar_config)
chat_agent = ChatAgent(firewall=FirewallStack(adapter=adapter, logger=logger), logger=logger)

logger.log('session_start', {'case_id': CASE_ID, 'pillar': PILLAR, 'mode': MODE})
print('Pipeline initialized')

## 5. Helper Function

In [ ]:
def run_question(question: str, trace_id: str):
    """Run one question through: team construction -> specialists -> compare -> synthesize."""
    logger.set_trace(trace_id)

    # Team construction
    team_fw = FirewallStack(adapter=adapter, logger=logger)
    tc = TeamConstructor(firewall=team_fw, logger=logger, catalog=catalog)
    selected = tc.select_specialists(
        question=question, pillar=PILLAR,
        available_specialists=list_domain_skills(),
        active_specialists=registry.list_active(),
        mode=MODE,
    )
    print(f'Selected: {selected}')
    print(f'(Warm: {[a["domain"] for a in registry.list_active()]})')
    print()

    # Run specialists (get_or_create reuses warm instances)
    outputs = {}
    for domain in selected:
        skill = load_domain_skill(domain)
        if skill is None:
            continue
        spec_config = pillar_loader.get_specialist_config(PILLAR, domain) or {}
        fw = FirewallStack(adapter=adapter, logger=logger)
        agent = registry.get_or_create(domain, PILLAR, skill, spec_config, fw, logger)
        output = agent.run(question, mode=MODE)
        outputs[domain] = output
        print(f'  {domain}: {output.findings[:200]}...')
    print()

    # Compare + synthesize
    review = general.compare(outputs, question)
    final = orchestrator.synthesize(outputs, review, question, MODE)

    logger.log('final_output', {'question': question, 'specialists': final.specialists_consulted})
    logger.clear_trace()
    return selected, outputs, review, final

## 6. Q1 — Positive Events Frequency

In [ ]:
selected_q1, outputs_q1, review_q1, final_q1 = run_question(QUESTION, 'q-001')

print('='*60)
print(f'Q1: {QUESTION}')
print('='*60)
print(chat_agent.format_for_reviewer(final_q1))

## 7. Follow-Up — Brings in Bureau

"Are the positive events consistent with the bureau scores?" — this question needs BOTH modeling (positive_events) AND bureau (scores). The team constructor should pull in bureau on top of the warm modeling specialist from Q1.

In [ ]:
# Preview data relevant to the follow-up: positive_events (from modeling) and bureau scores
print('=== Data for follow-up validation ===\n')

print('Bureau scores (from bureau table):')
bureau_rows = gw.query('bureau')
if bureau_rows:
    for row in bureau_rows[:2]:
        print(f'  fico={row.get(\"fico_score\")}, sbfe={row.get(\"sbfe_score\")}, ln_credit={row.get(\"ln_credit_score\")}, paydex={row.get(\"paydex_score\")}')

print('\nPositive events (from model_scores, sorted by trans_month):')
model_rows = gw.query('model_scores')
if model_rows:
    filtered = [(r.get('trans_month'), r.get('positive_events')) for r in model_rows]
    for tm, pe in filtered:
        print(f'  {tm}: positive_events={pe}')

In [ ]:
FOLLOW_UP = "Are the positive events consistent with the bureau scores?"
selected_fu, outputs_fu, review_fu, final_fu = run_question(FOLLOW_UP, 'q-002')

print('='*60)
print(f'FOLLOW-UP: {FOLLOW_UP}')
print('='*60)
print(chat_agent.format_for_reviewer(final_fu))

## 8. Q2 — Independent Topic (Recent Payment Pattern)

A completely different question — not related to positive events. The team constructor should select spend_payments, possibly bureau or crossbu.

In [ ]:
# Preview payment data for Q2 validation
print('=== Data for Q2 validation ===\n')

print('Monthly transactions (txn_monthly):')
txn = gw.query('txn_monthly')
if txn:
    # Sort by date descending to show most recent first
    sorted_txn = sorted(txn, key=lambda r: r.get('month', ''), reverse=True)
    for row in sorted_txn[:5]:
        print(f'  {row.get(\"month\")}: spend_total=${row.get(\"spend_total\")}, txn_count={row.get(\"txn_count\")}, category={row.get(\"category\")}')

print('\nRecent transaction-level payments (pmts_detail, last 5 by date):')
pmts = gw.query('pmts_detail')
if pmts:
    sorted_pmts = sorted(pmts, key=lambda r: r.get('date', ''), reverse=True)
    for row in sorted_pmts[:5]:
        print(f'  {row.get(\"date\")} ({row.get(\"month\")}): ${row.get(\"amount\")} @ {row.get(\"merchant_name\")}, risk={row.get(\"merchant_risk_score\")}')

In [ ]:
QUESTION_2 = "What does the recent 3 months payment pattern look like? Any signs of deterioration?"
selected_q2, outputs_q2, review_q2, final_q2 = run_question(QUESTION_2, 'q-003')

print('='*60)
print(f'Q2: {QUESTION_2}')
print('='*60)
print(chat_agent.format_for_reviewer(final_q2))

## 9. Session Info

In [ ]:
print('Active specialists across the session:')
for a in registry.list_active():
    print(f'  {a["domain"]}: {a["questions_answered"]} questions answered')

print()
print(f'Q1 selected: {selected_q1}')
print(f'Follow-up selected: {selected_fu}')
print(f'Q2 selected: {selected_q2}')

logger.log('session_end', {})
print('\nSession complete.')